In [41]:
import os
import traceback
import pdb #debug
import time
import shutil
import math
import re
import csv
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

import networkx as nx
#import pydot #deprecated
import pygraphviz as pgv
from networkx.drawing.nx_agraph import read_dot, from_agraph
from graphviz import Digraph
from graphviz import Source

#Neurmorphology and Simulation
import neuroarch.na as na


from IPython.display import display, JSON
import json
from scipy import stats

# ANSI color codes
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"

In [9]:
#turn on breakpoints?
%pdb on

Automatic pdb calling has been turned ON


In [2]:
!git init
!git remote add origin https://github.com/sgarnell/archive

Reinitialized existing Git repository in /home/ffbo/ffbo/.git/
fatal: remote origin already exists.


In [3]:
!git remote -v

origin	https://github.com/sgarnell/archive.git (fetch)
origin	https://github.com/sgarnell/archive.git (push)


In [4]:
# Stage the notebook file
!git add 'PC_Latex_Generator_v1.ipynb'

# Commit the changes
!git commit -m "Auto commit from Jupyter for {notebook_name}"

# Push to GitHub
!git push origin fbl2-main  # Replace 'main' with 'master' if needed

[fbl2-main 906555e] Auto commit from Jupyter for {notebook_name}
 1 file changed, 237 insertions(+)
 create mode 100644 PC_Latex_Generator_v1.ipynb
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 16 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 2.55 KiB | 2.55 MiB/s, done.
Total 3 (delta 1), reused 0 (delta 0)
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/sgarnell/archive.git
   e1d97ae..906555e  fbl2-main -> fbl2-main


In [93]:
import os
import json
import re

#Load data from 3 files provided
#  - Dot file with graphviz graph
#  - nt_file contains the neurotransmitter data for each neuron
#  - Motif contains data about specific neuron combinations, ie Wilson-Cowan
def load_files(dot_file, nt_file, motif_file):
    with open(dot_file, "r") as f:
        dot_lines = f.readlines()
    with open(nt_file, "r") as f:
        nt_dict = json.load(f)
    with open(motif_file, "r") as f:
        motif_dict = json.load(f)
        
    return dot_lines, nt_dict, motif_dict


def get_polarity(neuron, nt_dict, polarity_dict):
    nts = nt_dict.get(neuron, {})
    for nt in nts:
        sign = polarity_dict.get(nt.lower())
        if sign is not None:
            return "+" if sign == 1 else "-"
    return "?"

def extract_prefix(name, cutoff):
    return name[:cutoff]

def extract_edges(dot_lines):
    edges = []
    for line in dot_lines:
        match = re.search(r'"(.+?)" -> "(.+?)" \[label="Zscore = ([0-9.]+)"', line)
        if match:
            head, tail, zscore = match.groups()
            edges.append((head, tail, float(zscore)))
            
    return edges

def get_synapse_name(head, tail):
    if "--" in head:
        return head
    if "--" in tail:
        return tail
    return None

def abstract_synapse_name(synapse, cutoff):
    if "--" not in synapse:
        return synapse
    source, target = synapse.split("--")
    return f"{source[:cutoff]}-{target[:cutoff]}"



def get_wc_neurons(motif_dict):
    wc_neurons = set()
    if motif_dict:
        for motif in motif_dict.values():
            if abbreviate_motif_type(motif.get("motifType", "unknown")).upper() == "WC":
                wc_neurons.add(motif["neuron_A"])
                wc_neurons.add(motif["neuron_B"])
    return wc_neurons

def find_best_parallel_cutoff(edges, min_cutoff=3, max_cutoff=20, exclude_neurons=None):
    if exclude_neurons is None:
        exclude_neurons = set()

    cutoff_scores = {}
    cutoff_groups = {}

    for cutoff in range(min_cutoff, max_cutoff + 1):
        groups = defaultdict(list)
        for head, tail, z in edges:
            if head in exclude_neurons or tail in exclude_neurons:
                continue
            key = f"{extract_prefix(head, cutoff)}->{extract_prefix(tail, cutoff)}"
            groups[key].append((head, tail, z))

        valid_groups = [g for g in groups.values() if len(g) > 1]
        group_count = len(valid_groups)
        total_members = sum(len(g) for g in valid_groups)
        avg_group_size = total_members / group_count if group_count > 0 else 0

        cutoff_scores[cutoff] = {
            "group_count": group_count,
            "total_members": total_members,
            "avg_group_size": avg_group_size
        }
        cutoff_groups[cutoff] = groups

    # Step 2: Detect plateau in avg_group_size
    plateau_cutoffs = []
    for cutoff in range(min_cutoff + 1, max_cutoff):
        prev_avg = cutoff_scores[cutoff - 1]["avg_group_size"]
        curr_avg = cutoff_scores[cutoff]["avg_group_size"]
        next_avg = cutoff_scores[cutoff + 1]["avg_group_size"]
        if abs(curr_avg - prev_avg) < 1.0 and abs(curr_avg - next_avg) < 1.0:
            plateau_cutoffs.append(cutoff)

    # Step 3: Choose smallest cutoff in plateau, fallback to max total_members
    if plateau_cutoffs:
        best_cutoff = min(plateau_cutoffs)
    else:
        best_cutoff = max(cutoff_scores, key=lambda k: cutoff_scores[k]["total_members"])

    return cutoff_groups[best_cutoff], best_cutoff


def exclude_motif_edges(edges, motifs_dict):
    motif_neurons = set(motifs_dict.keys())
    filtered = []
    for head, tail, z in edges:
        synapse = get_synapse_name(head, tail)
        reverse_synapse = None
        if synapse and "--" in synapse:
            a, b = synapse.split("--")
            reverse_synapse = f"{b}--{a}"

        if (
            head in motif_neurons or
            tail in motif_neurons or
            synapse in motifs_dict or
            reverse_synapse in motifs_dict
        ):
            continue

        filtered.append((head, tail, z))
    return filtered



def build_neuron_equations(edges, nt_dict, polarity_dict, motif_dict=None):
    
    
    used_targets = set()
    used_sources = set()
    
    #remove all known Motifs from the edge data
    xmotif_edges = exclude_motif_edges(edges, motif_dict)
    equations = []
    wc_neurons = get_wc_neurons(motif_dict)
    
    def latex_clean(name):
        return (
            name.replace("_", "")
                .replace("--", "-")
                .replace("(", "")
                .replace(")", "")
                .replace(">", "to")
        )


    # Step 1: Build recurrent motif equations
    motif_groups = {}
    edge_set = set((head, tail) for head, tail, _ in edges)

    def extract_type_and_region(neuron):
        match = re.match(r"([A-Za-z0-9_]+)\(([^)]+)\)(\d*)", neuron)
        if match:
            typ, region, index = match.groups()
            return typ, region
        return None, None

    # Group edges by neuron type and region pairs
    for head, tail, z in edges:
        src_type, src_region = extract_type_and_region(head)
        tgt_type, tgt_region = extract_type_and_region(tail)
        if not src_type or not tgt_type:
            continue

        key = tuple(sorted([(src_type, src_region), (tgt_type, tgt_region)]))
        motif_groups.setdefault(key, []).append((head, tail, z))

    equations = []
    for key, group in motif_groups.items():
        # Check if reciprocal edges exist
        has_forward = any((head, tail) in edge_set for head, tail, _ in group)
        has_reverse = any((tail, head) in edge_set for head, tail, _ in group)

        if not (has_forward and has_reverse):
            continue  # skip this motif — not truly recurrent

        #We now can udate the groups into the used set to avoid duplicaiton
        for head, tail, _ in group:
            used_sources.add(head)
            used_targets.add(tail)
            used_sources.add(tail)  # because tail might also be a source in reverse edge
            used_targets.add(head)  # same logic for head

        
        (a_type, a_region), (b_type, b_region) = key
        
        motif_label = f"\\mathcal{{R}}_{{{a_type} \\leftrightarrow {b_type}}}^{{({a_region},{b_region})}}"
        eq = (
            "\\begin{align*}\n"
            f"{motif_label} = & \\sum_{{i,j}} \\bm{{W}}_{{{a_type}({a_region})_i \\leftarrow {b_type}({b_region})_j}} "
            f"\\cdot \\mu_{{{b_type}({b_region})_j}}^{{(A)}} \\\\\n"
            f"+ & \\bm{{W}}_{{{b_type}({b_region})_j \\leftarrow {a_type}({a_region})_i}} "
            f"\\cdot \\mu_{{{a_type}({a_region})_i}}^{{(A)}}\n"
            "\\end{align*}"
        )


        equations.append(eq)

        
    # Step 2: Synapse abstraction using cutoff
    parallel_groups, cutoff_used = find_best_parallel_cutoff(edges, exclude_neurons=wc_neurons)

    # Build abstracted synapse groups using cutoff
    abstracted_groups = defaultdict(list)
    for head, tail, z in xmotif_edges:
        synapse = get_synapse_name(head, tail)
        if synapse:
            abstracted_key = abstract_synapse_name(synapse, cutoff_used)
            abstracted_groups[abstracted_key].append((head, tail, z))

    # Emit ensemble equations
    for key, group in abstracted_groups.items():
        if len(group) < 2:
            continue
        ensemble_label = f"\\mu^{{\\epsilon_{{{latex_clean(key)}}}^{{(P)}}}}"
        terms = []
        for head, tail, z in group:
            polarity = get_polarity(head, nt_dict, polarity_dict)
            term = (
                f"\\left( z^{{{z:.2f}}} \\cdot W^{{({polarity})}}_{{{latex_clean(tail)} \\leftarrow {latex_clean(head)}}} "
                f"\\cdot \\mu_{{{latex_clean(head)}}}^{{(A)}} \\right)"
            )
            terms.append(term)
            used_targets.add(tail)
            used_sources.add(head)
        eq = (
            f"% Cutoff used: {cutoff_used}\n"
            "\\begin{align*}\n"
            f"{ensemble_label} = & " + " \\\\\n+ & ".join(terms) + "\n"
            "\\end{align*}"
        )
        equations.append(eq)


    fan_in_map = defaultdict(list)
    for head, tail, z in xmotif_edges:
        if tail in wc_neurons or tail in used_targets:
            continue
        if "--" in tail or "--" in head:
            continue
        fan_in_map[tail].append((head, z))

    for target, sources in fan_in_map.items():
        if len(sources) < 2:
            continue
        ensemble_label = f"\\mu^\\epsilon_{{{target}}}^{{(P)}}"
        terms = []
        for head, z in sources:
            polarity = get_polarity(head, nt_dict, polarity_dict)
            term = (
                f"\\bm{{W}}_{{{target} \\leftarrow {head}}}^{{({polarity})}}(z^{{{z:.2f}}}) "
                f"\\cdot \\mu_{{{head}}}^{{(A)}}"
            )
            terms.append(term)
            used_sources.add(head)
        eq = (
            "\\begin{align*}\n"
            f"{ensemble_label} = & " + " \\\\\n+ & ".join(terms) + "\n"
            "\\end{align*}"
        )
        equations.append(eq)
        used_targets.add(target)

    # Step 4: Fan-out ensembles (one source → many targets)
    fan_out_map = defaultdict(list)
    for head, tail, z in xmotif_edges:
        if head in wc_neurons or head in used_sources:
            continue
        if "--" in tail or "--" in head:
            continue
        fan_out_map[head].append((tail, z))

    for source, targets in fan_out_map.items():
        if len(targets) < 2:
            continue
        ensemble_label = f"\\mu^\\epsilon_{{{source}}}^{{(A)}}"
        terms = []
        for tail, z in targets:
            polarity = get_polarity(source, nt_dict, polarity_dict)
            term = (
                f"\\bm{{W}}_{{{tail} \\leftarrow {source}}}^{{({polarity})}}(z^{{{z:.2f}}}) "
                f"\\cdot \\mu_{{{source}}}^{{(A)}}"
            )
            terms.append(term)
            used_targets.add(tail)
        eq = (
            "\\begin{align*}\n"
            f"{ensemble_label} = & " + " \\\\\n+ & ".join(terms) + "\n"
            "\\end{align*}"
        )
        equations.append(eq)

    # Step 5: Remaining microstates
    for head, tail, z in xmotif_edges:
        if tail in used_targets or head in used_sources:
            continue
        polarity = get_polarity(head, nt_dict, polarity_dict)
        eq = (
            f"\\mu_{{{tail}}}^{{(P)}} = "
            f"\\bm{{W}}_{{{tail} \\leftarrow {head}}}^{{({polarity})}}(z^{{{z:.2f}}}) "
            f"\\cdot \\mu_{{{head}}}^{{(A)}}"
        )
        equations.append(eq)

    return equations



def abbreviate_motif_type(motif_type):
    words = re.split(r'[^a-zA-Z0-9]+', motif_type)
    return ''.join(word[0].upper() for word in words if word)




def build_motif_equations(motif_dict, nt_dict, polarity_dict):
    equations = []

    for motif in motif_dict.values():
        motif_type = motif.get("motifType", "unknown")
        if abbreviate_motif_type(motif_type).upper() != "WC":
            continue  # Only handle WC motifs here

        A = motif["neuron_A"]
        B = motif["neuron_B"]
        z_A = motif.get("Z_A", 1.0)
        z_B = motif.get("Z_B", 1.0)
        polarity_A = get_polarity(A, nt_dict, polarity_dict)
        polarity_B = get_polarity(B, nt_dict, polarity_dict)
        abb_mt = abbreviate_motif_type(motif_type)
        motif_label = f"{abb_mt.upper()}_{{{A},{B}}}"

        eq = (
            "\\begin{align*}\n"
            f"\\mu_{{{motif_label}}}^{{(P)}} = & \\bm{{W}}_{{{motif_label} \\leftarrow {A}}}^{{({polarity_A})}}(z^{{{z_A:.2f}}}) "
            f"\\cdot \\mu_{{{A}}}^{{(A)}} \\\\\n"
            f"+ & \\bm{{W}}_{{{motif_label} \\leftarrow {B}}}^{{({polarity_B})}}(z^{{{z_B:.2f}}}) "
            f"\\cdot \\mu_{{{B}}}^{{(A)}}\n"
            "\\end{align*}"
        )
        equations.append(eq)

    return equations




# def build_recurrent_motif_equations(edges):
    
#     motif_groups = {}
#     edge_set = set((head, tail) for head, tail, _ in edges)

#     def extract_type_and_region(neuron):
#         match = re.match(r"([A-Za-z0-9_]+)\(([^)]+)\)(\d*)", neuron)
#         if match:
#             typ, region, index = match.groups()
#             return typ, region
#         return None, None

#     # Group edges by neuron type and region pairs
#     for head, tail, z in edges:
#         src_type, src_region = extract_type_and_region(head)
#         tgt_type, tgt_region = extract_type_and_region(tail)
#         if not src_type or not tgt_type:
#             continue

#         key = tuple(sorted([(src_type, src_region), (tgt_type, tgt_region)]))
#         motif_groups.setdefault(key, []).append((head, tail, z))

#     equations = []
#     for key, group in motif_groups.items():
#         # Check if reciprocal edges exist
#         has_forward = any((head, tail) in edge_set for head, tail, _ in group)
#         has_reverse = any((tail, head) in edge_set for head, tail, _ in group)

#         if not (has_forward and has_reverse):
#             continue  # skip this motif — not truly recurrent

#         (a_type, a_region), (b_type, b_region) = key
        
#         motif_label = f"\\mathcal{{R}}_{{{a_type} \\leftrightarrow {b_type}}}^{{({a_region},{b_region})}}"
#         eq = (
#             "\\begin{align*}\n"
#             f"{motif_label} = & \\sum_{{i,j}} \\bm{{W}}_{{{a_type}({a_region})_i \\leftarrow {b_type}({b_region})_j}} "
#             f"\\cdot \\mu_{{{b_type}({b_region})_j}}^{{(A)}} \\\\\n"
#             f"+ & \\bm{{W}}_{{{b_type}({b_region})_j \\leftarrow {a_type}({a_region})_i}} "
#             f"\\cdot \\mu_{{{a_type}({a_region})_i}}^{{(A)}}\n"
#             "\\end{align*}"
#         )


#         equations.append(eq)

#     return equations




def write_latex(equations, filename):
    with open(filename, "w") as f:
        f.write("\\section*{Predictive Coding Equations}\n\n")
        for eq in equations:
            stripped = eq.strip()
            if stripped.startswith("\\begin{align") or stripped.startswith("\\begin{multline") or "\\begin{align" in stripped:
                f.write(eq + "\n\n")
            else:
                f.write("\\begin{equation}\n")
                f.write(eq + "\n")
                f.write("\\end{equation}\n\n")


                             

def main():
    folder = "PC_LatexGen"
    dot_file = os.path.join(folder, "MBON03_prePost_merged_filtered.gv")
    nt_file = os.path.join(folder, "neuronNTdict.json")
    motif_file = os.path.join(folder, "motifs.json")
    

    polarity_dict = {
        "acetylcholine": 1,
        "glutamate": 0,
        "gaba": 0,
        "dopamine": 1,
        "octopamine": 1,
        "serotonin": 0
    }
    


    #Get data from all files
    dot_lines, nt_dict, motif_dict = load_files(dot_file, nt_file, motif_file)
    
    #get edges
    edges = extract_edges(dot_lines)
    
    #Detect reciprocal connections between neuron families
    #Group them by shared neuropil (e.g., PB06b ↔ PB08)
    #Generate a condensed LaTeX block
    # recurrent_eqs = build_recurrent_motif_equations(edges)
    
    #get equations
    neuron_eqs = build_neuron_equations(edges, nt_dict, polarity_dict,motif_dict)

    #get motifs
    motif_eqs = build_motif_equations(motif_dict, nt_dict, polarity_dict)

    all_eqs = neuron_eqs + motif_eqs
    
    output_path = os.path.join(folder, "predictive_coding_equations.tex")
    write_latex(all_eqs, output_path)
    print(f"LaTeX file written to: {output_path}")

if __name__ == "__main__":
    main()




LaTeX file written to: PC_LatexGen/predictive_coding_equations.tex
